In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import joblib

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df = pd.read_csv("../data/processed/mumbai_house_price_cleaned.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (51767, 14)


In [3]:
print("Columns in dataset:")

for i, column in enumerate(df.columns, start=1):
    print(i, ":", column)

Columns in dataset:
1 : price
2 : area
3 : price_per_sqft
4 : locality
5 : city
6 : property_type
7 : bedroom_num
8 : bathroom_num
9 : balcony_num
10 : furnished
11 : age
12 : total_floors
13 : latitude
14 : longitude


In [4]:
df.head()

,price,area,price_per_sqft,locality,city,property_type,bedroom_num,bathroom_num,balcony_num,furnished,age,total_floors,latitude,longitude
0,6600283,757,8719.000000,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,0,1,19.244410,73.123253
1,6169841,652,9462.946319,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,0,1,19.257294,73.148872
2,4599936,396,11616.000000,Dombivali,Mumbai,Apartment,1,1,0,Unfurnished,0,1,19.209026,73.081276
3,51980000,1130,46000.000000,Ville Parle,Mumbai,Apartment,3,3,0,Unfurnished,0,1,19.097841,72.851158
4,3915000,435,9000.000000,Nala Sopara,Mumbai,Apartment,1,1,0,Unfurnished,0,1,19.420601,72.809319


In [5]:
df.dtypes

price               int64
area                int64
price_per_sqft    float64
locality              str
city                  str
property_type         str
bedroom_num         int64
bathroom_num        int64
balcony_num         int64
furnished             str
age                 int64
total_floors        int64
latitude          float64
longitude         float64
dtype: object

In [6]:
missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values)

Missing values:
price             0
area              0
price_per_sqft    0
locality          0
city              0
property_type     0
bedroom_num       0
bathroom_num      0
balcony_num       0
furnished         0
age               0
total_floors      0
latitude          0
longitude         0
dtype: int64


In [7]:
categorical_check_columns = [
    "locality",
    "city",
    "property_type",
    "furnished"
]

for column in categorical_check_columns:
    if column in df.columns:
        print(f"\n{column}")
        print("Unique values:", df[column].nunique())
        print(df[column].unique()[:20])


locality
Unique values: 408
<StringArray>
[       'Kalyan',     'Dombivali',   'Ville Parle',   'Nala Sopara',
          'Ulwe',    'Jogeshwari',       'Chembur',         'Nerul',
        'Mulund', 'Nallasopara W',      'Dombivli',        'Taloje',
        'Panvel',    'Shil Phata',       'Vangani',      'Kharghar',
    'Nalasopara',      'Badlapur',        'Mumbra',         'Vasai']
Length: 20, dtype: str

city
Unique values: 4
<StringArray>
['Mumbai', 'Mumbai South', 'Central Mumbai', 'Western Mumbai']
Length: 4, dtype: str

property_type
Unique values: 5
<StringArray>
[        'Apartment',             'Villa', 'Independent House',
 'Independent Floor',  'Studio Apartment']
Length: 5, dtype: str

furnished
Unique values: 3
<StringArray>
['Unfurnished', 'Semi-Furnished', 'Furnished']
Length: 3, dtype: str


In [8]:
if "city" in df.columns:
    print("Unique cities:")
    print(df["city"].value_counts())
else:
    print("City column not found")

Unique cities:
city
Mumbai            50711
Western Mumbai      489
Central Mumbai      362
Mumbai South        205
Name: count, dtype: int64


In [9]:
df["area_per_bedroom"] = (
    df["area"] / df["bedroom_num"].replace(0, np.nan)
)

df["area_per_bedroom"] = df["area_per_bedroom"].replace(
    [np.inf, -np.inf], np.nan
)

print("area_per_bedroom feature created")
df[["area", "bedroom_num", "area_per_bedroom"]].head()

area_per_bedroom feature created


,area,bedroom_num,area_per_bedroom
0,757,2,378.500000
1,652,2,326.000000
2,396,1,396.000000
3,1130,3,376.666667
4,435,1,435.000000


In [10]:
df["bathroom_per_bedroom"] = (
    df["bathroom_num"] / df["bedroom_num"].replace(0, np.nan)
)

df["bathroom_per_bedroom"] = df["bathroom_per_bedroom"].replace(
    [np.inf, -np.inf], np.nan
)

print("bathroom_per_bedroom feature created")
df[["bedroom_num", "bathroom_num", "bathroom_per_bedroom"]].head()

bathroom_per_bedroom feature created


,bedroom_num,bathroom_num,bathroom_per_bedroom
0,2,2,1.0
1,2,2,1.0
2,1,1,1.0
3,3,3,1.0
4,1,1,1.0


In [11]:
def age_group(age):
    if pd.isna(age):
        return "Unknown"
    elif age <= 5:
        return "New"
    elif age <= 15:
        return "Moderately New"
    elif age <= 30:
        return "Old"
    else:
        return "Very Old"


df["age_group"] = df["age"].apply(age_group)

print("Age groups created")
print(df["age_group"].value_counts())

Age groups created
age_group
New               43090
Moderately New     7541
Old                1070
Very Old             66
Name: count, dtype: int64


In [12]:
new_features = [
    "area_per_bedroom",
    "bathroom_per_bedroom",
    "age_group"
]

df[new_features].head(10)

,area_per_bedroom,bathroom_per_bedroom,age_group
0,378.500000,1.00,New
1,326.000000,1.00,New
2,396.000000,1.00,New
3,376.666667,1.00,New
4,435.000000,1.00,New
5,720.000000,1.00,New
6,326.000000,1.00,New
7,282.500000,1.00,New
8,590.000000,1.00,New
9,650.000000,1.25,New


In [13]:
target_column = "price"

y = df[target_column]

print("Target variable:", target_column)
print("Target shape:", y.shape)
print("Target mean:", y.mean())
print("Target median:", y.median())

Target variable: price
Target shape: (51767,)
Target mean: 20298373.60839917
Target median: 12000000.0


In [14]:
X = df.drop(columns=[target_column])

# price_per_sqft is derived from price.
# Therefore, it must not be used as an input feature.

if "price_per_sqft" in X.columns:
    X = X.drop(columns=["price_per_sqft"])

print("Features after removing target and price_per_sqft:")
print(X.columns.tolist())

Features after removing target and price_per_sqft:
['area', 'locality', 'city', 'property_type', 'bedroom_num', 'bathroom_num', 'balcony_num', 'furnished', 'age', 'total_floors', 'latitude', 'longitude', 'area_per_bedroom', 'bathroom_per_bedroom', 'age_group']


In [15]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (51767, 15)
Target shape: (51767,)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target :", y_test.shape)

Training data: (41413, 15)
Testing data : (10354, 15)
Training target: (41413,)
Testing target : (10354,)


In [17]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['area', 'bedroom_num', 'bathroom_num', 'balcony_num', 'age', 'total_floors', 'latitude', 'longitude', 'area_per_bedroom', 'bathroom_per_bedroom']

Categorical features:
['locality', 'city', 'property_type', 'furnished', 'age_group']


C:\Users\Tejas Dalvi\AppData\Local\Temp\ipykernel_21516\1124402278.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [18]:
if "city" in X_train.columns:
    
    print("Unique city values:")
    print(X_train["city"].nunique())
    print(X_train["city"].unique())

    if X_train["city"].nunique() <= 1:
        X_train = X_train.drop(columns=["city"])
        X_test = X_test.drop(columns=["city"])

        print("\nCity column removed because it contains only one unique value.")
    else:
        print("\nCity column retained because multiple cities exist.")

Unique city values:
4
<StringArray>
['Mumbai', 'Western Mumbai', 'Central Mumbai', 'Mumbai South']
Length: 4, dtype: str

City column retained because multiple cities exist.


In [19]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Final numerical features:")
print(numerical_features)

print("\nFinal categorical features:")
print(categorical_features)

Final numerical features:
['area', 'bedroom_num', 'bathroom_num', 'balcony_num', 'age', 'total_floors', 'latitude', 'longitude', 'area_per_bedroom', 'bathroom_per_bedroom']

Final categorical features:
['locality', 'city', 'property_type', 'furnished', 'age_group']


C:\Users\Tejas Dalvi\AppData\Local\Temp\ipykernel_21516\3162751533.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [20]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print("Numerical preprocessing pipeline created")

Numerical preprocessing pipeline created


In [21]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=30
            )
        )
    ]
)

print("Categorical preprocessing pipeline created")

Categorical preprocessing pipeline created


In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("ColumnTransformer created successfully")

ColumnTransformer created successfully


In [23]:
X_train_processed = preprocessor.fit_transform(X_train)

print("Training data preprocessing completed")
print("Processed training shape:", X_train_processed.shape)

Training data preprocessing completed
Processed training shape: (41413, 112)


In [24]:
X_test_processed = preprocessor.transform(X_test)

print("Testing data preprocessing completed")
print("Processed testing shape:", X_test_processed.shape)

Testing data preprocessing completed
Processed testing shape: (10354, 112)


In [25]:
print("Training data type:", type(X_train_processed))
print("Testing data type :", type(X_test_processed))

Training data type: <class 'scipy.sparse._csr.csr_matrix'>
Testing data type : <class 'scipy.sparse._csr.csr_matrix'>


In [26]:
feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(feature_names))

print("\nFirst 50 processed features:")

for feature in feature_names[:50]:
    print(feature)

Total processed features: 112

First 50 processed features:
num__area
num__bedroom_num
num__bathroom_num
num__balcony_num
num__age
num__total_floors
num__latitude
num__longitude
num__area_per_bedroom
num__bathroom_per_bedroom
cat__locality_Agripada
cat__locality_Airoli
cat__locality_Ambernath
cat__locality_Andheri
cat__locality_Anjurdive
cat__locality_Badlapur
cat__locality_Bandra
cat__locality_Belapur
cat__locality_Bhandup
cat__locality_Bhayandar
cat__locality_Bhiwandi
cat__locality_Boisar
cat__locality_Borivali
cat__locality_Byculla
cat__locality_Chembur
cat__locality_Colaba
cat__locality_Dadar
cat__locality_Dahisar
cat__locality_Deonar
cat__locality_Dombivali
cat__locality_Dombivli
cat__locality_Dronagiri
cat__locality_Ghansoli
cat__locality_Ghatkopar
cat__locality_Girgaon
cat__locality_Goregaon
cat__locality_Hiranandani Estates
cat__locality_Jogeshwari
cat__locality_Juhu
cat__locality_Kalamboli
cat__locality_Kalwa
cat__locality_Kalyan
cat__locality_Kamothe
cat__locality_Kandivali
c

In [27]:
sample_size = min(10, X_train_processed.shape[0])

sample_processed = X_train_processed[:sample_size].toarray()

processed_sample_df = pd.DataFrame(
    sample_processed,
    columns=feature_names
)

processed_sample_df.head()

,num__area,num__bedroom_num,num__bathroom_num,num__balcony_num,num__age,num__total_floors,num__latitude,num__longitude,num__area_per_bedroom,num__bathroom_per_bedroom,...,cat__property_type_Independent House,cat__property_type_Villa,cat__property_type_infrequent_sklearn,cat__furnished_Furnished,cat__furnished_Semi-Furnished,cat__furnished_Unfurnished,cat__age_group_Moderately New,cat__age_group_New,cat__age_group_Old,cat__age_group_Very Old
0,0.632573,1.065551,-0.116667,-0.277509,1.273837,-0.012307,-0.244567,-0.205871,-0.164267,-1.291428,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,-0.588485,-1.069934,-1.267309,-0.277509,-0.541006,-0.012307,-0.381386,0.426457,0.568347,-0.333726,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,1.898304,1.065551,1.033975,-0.277509,-0.541006,-0.012307,19.946263,8.069459,1.667269,-0.333726,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-0.165582,-0.002192,-0.116667,-0.277509,-0.541006,-0.012307,-0.029424,-0.036721,-0.388361,-0.333726,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0.558118,1.065551,1.033975,-0.277509,2.181259,-0.012307,0.021142,-0.278162,-0.272004,-0.333726,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [28]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.joblib"
)

print("Preprocessor saved successfully")

Preprocessor saved successfully


In [30]:
joblib.dump(
    X_train_processed,
    "../models/X_train_processed.joblib"
)

joblib.dump(
    X_test_processed,
    "../models/X_test_processed.joblib"
)

joblib.dump(
    y_train,
    "../models/y_train.joblib"
)

joblib.dump(
    y_test,
    "../models/y_test.joblib"
)

print("Prepared train/test data saved successfully")

Prepared train/test data saved successfully


In [31]:
import os

files_to_check = [
    "../models/preprocessor.joblib",
    "../models/X_train_processed.joblib",
    "../models/X_test_processed.joblib",
    "../models/y_train.joblib",
    "../models/y_test.joblib"
]

for file in files_to_check:
    if os.path.exists(file):
        print("✓", file)
    else:
        print("✗", file)

✓ ../models/preprocessor.joblib
✓ ../models/X_train_processed.joblib
✓ ../models/X_test_processed.joblib
✓ ../models/y_train.joblib
✓ ../models/y_test.joblib


In [32]:
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print("\nOriginal dataset shape:", df.shape)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

print("\nOriginal input features:", X.shape[1])
print("Processed features:", X_train_processed.shape[1])

print("\nNumerical features:")
for feature in numerical_features:
    print(" -", feature)

print("\nCategorical features:")
for feature in categorical_features:
    print(" -", feature)

print("\nTarget variable:", target_column)

print("\nExcluded from model:")
print(" - price")
print(" - price_per_sqft (target leakage)")

print("\nFeature engineering completed successfully.")

FEATURE ENGINEERING SUMMARY

Original dataset shape: (51767, 17)

Training samples: 41413
Testing samples : 10354

Original input features: 15
Processed features: 112

Numerical features:
 - area
 - bedroom_num
 - bathroom_num
 - balcony_num
 - age
 - total_floors
 - latitude
 - longitude
 - area_per_bedroom
 - bathroom_per_bedroom

Categorical features:
 - locality
 - city
 - property_type
 - furnished
 - age_group

Target variable: price

Excluded from model:
 - price
 - price_per_sqft (target leakage)

Feature engineering completed successfully.


In [33]:
print("""
FEATURE ENGINEERING COMPLETED

1. Loaded the cleaned Mumbai house price dataset.
2. Created additional features:
   - area_per_bedroom
   - bathroom_per_bedroom
   - age_group
3. Separated the target variable 'price'.
4. Removed 'price_per_sqft' to prevent target leakage.
5. Split the dataset into training and testing sets.
6. Identified numerical and categorical features.
7. Applied median imputation and standard scaling to numerical features.
8. Applied one-hot encoding to categorical features.
9. Used handle_unknown='ignore' for unseen categorical values.
10. Saved the preprocessing pipeline and processed datasets.

The dataset is now ready for machine learning model training.
""")


FEATURE ENGINEERING COMPLETED

1. Loaded the cleaned Mumbai house price dataset.
2. Created additional features:
   - area_per_bedroom
   - bathroom_per_bedroom
   - age_group
3. Separated the target variable 'price'.
4. Removed 'price_per_sqft' to prevent target leakage.
5. Split the dataset into training and testing sets.
6. Identified numerical and categorical features.
7. Applied median imputation and standard scaling to numerical features.
8. Applied one-hot encoding to categorical features.
9. Used handle_unknown='ignore' for unseen categorical values.
10. Saved the preprocessing pipeline and processed datasets.

The dataset is now ready for machine learning model training.

